# **How to Build and Fine-Tune a Small Language Model**

*--A Step-by-Step Guide for Beginners, Researchers, and Non-Programmers*

1) Paper Version: https://www.amazon.com/dp/B0G3MYWTJK

2) PDF version: https://leanpub.com/howtobuildandfine-tuneasmalllanguagemodel

Reference:

Liu, J. Paul. 2025. How to Build and Fine-Tune a Small Language Model. Leanpub. 479 p.

# Chapter 5: Configuring Your Model Architecture.ipynb

## Part 1: Understanding Parameters

Remember from Chapter 2 when you built GPT? The model had **parameters**—weights and biases that get adjusted during training.

### What Counts as a Parameter?

In [ ]:
'''
# Simplified example from Chapter 2
class TinyGPT(nn.Module):
    def __init__(self, vocab_size=65, n_embd=384):
        super().__init__()
        # Token embedding: vocab_size × n_embd parameters
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        # 65 × 384 = 24,960 parameters

        # Position embedding: block_size × n_embd parameters
        self.position_embedding = nn.Embedding(256, n_embd)
        # 256 × 384 = 98,304 parameters

        # Attention: 3 × (n_embd × n_embd) parameters (Q, K, V)
        self.attention = nn.Linear(n_embd, 3 * n_embd)
        # 3 × 384 × 384 = 442,368 parameters

        # Feed-forward: 2 layers
        self.ffn = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # 384 × 1536 = 589,824
            nn.Linear(4 * n_embd, n_embd),  # 1536 × 384 = 589,824
        )

# Total for just ONE transformer block: ~1.7M parameters
# Multiply by number of layers: 6 layers × 1.7M = ~10M parameters
'''

'\n# Simplified example from Chapter 2\nclass TinyGPT(nn.Module):\n    def __init__(self, vocab_size=65, n_embd=384):\n        super().__init__()\n        # Token embedding: vocab_size × n_embd parameters\n        self.token_embedding = nn.Embedding(vocab_size, n_embd)\n        # 65 × 384 = 24,960 parameters\n\n        # Position embedding: block_size × n_embd parameters\n        self.position_embedding = nn.Embedding(256, n_embd)\n        # 256 × 384 = 98,304 parameters\n\n        # Attention: 3 × (n_embd × n_embd) parameters (Q, K, V)\n        self.attention = nn.Linear(n_embd, 3 * n_embd)\n        # 3 × 384 × 384 = 442,368 parameters\n\n        # Feed-forward: 2 layers\n        self.ffn = nn.Sequential(\n            nn.Linear(n_embd, 4 * n_embd),  # 384 × 1536 = 589,824\n            nn.Linear(4 * n_embd, n_embd),  # 1536 × 384 = 589,824\n        )\n\n# Total for just ONE transformer block: ~1.7M parameters\n# Multiply by number of layers: 6 layers × 1.7M = ~10M parameters\n'

## Part 2: The Four Key Configuration Decisions

Every model architecture requires four main decisions:

### 1. Hidden Size (n_embd or d_model)

In [ ]:
def recommend_hidden_size(task_complexity: str, data_size_mb: int) -> int:
    """Recommend hidden size based on task and data"""

    if task_complexity == "simple":  # Sentiment analysis, classification
        if data_size_mb < 50:
            return 384
        elif data_size_mb < 500:
            return 512
        else:
            return 768

    elif task_complexity == "medium":  # Summarization, simple QA
        if data_size_mb < 100:
            return 512
        elif data_size_mb < 1000:
            return 768
        else:
            return 1024

    elif task_complexity == "complex":  # Creative writing, complex reasoning
        if data_size_mb < 500:
            return 768
        elif data_size_mb < 5000:
            return 1024
        else:
            return 1280

    return 768  # Safe default

# Examples
print(f"Simple task, 20MB data: n_embd = {recommend_hidden_size('simple', 20)}")
print(f"Medium task, 200MB data: n_embd = {recommend_hidden_size('medium', 200)}")
print(f"Complex task, 2GB data: n_embd = {recommend_hidden_size('complex', 2000)}")

Simple task, 20MB data: n_embd = 384
Medium task, 200MB data: n_embd = 768
Complex task, 2GB data: n_embd = 1024


# 2. Number of Layers (n_layer or num_layers)

In [ ]:
def recommend_num_layers(n_embd: int, target_params_millions: int) -> int:
    """
    Calculate number of layers for target parameter count.

    Rough formula: params ≈ 12 × n_embd² × n_layer
    """
    params_per_layer = 12 * (n_embd ** 2)
    n_layer = int(target_params_millions * 1e6 / params_per_layer)

    # Constrain to reasonable range
    n_layer = max(6, min(48, n_layer))

    return n_layer

# Examples
print(f"125M params, n_embd=768: {recommend_num_layers(768, 125)} layers")
print(f"350M params, n_embd=1024: {recommend_num_layers(1024, 350)} layers")
print(f"1B params, n_embd=1280: {recommend_num_layers(1280, 1000)} layers")

125M params, n_embd=768: 17 layers
350M params, n_embd=1024: 27 layers
1B params, n_embd=1280: 48 layers


### 3. Number of Attention Heads (n_head)

In [ ]:
def recommend_num_heads(n_embd: int) -> int:
    """Recommend number of attention heads"""
    # Aim for 64 dimensions per head
    ideal_head_size = 64
    n_head = n_embd // ideal_head_size

    # Ensure divisibility
    while n_embd % n_head != 0:
        n_head -= 1

    # Constrain to reasonable range
    n_head = max(4, min(32, n_head))

    return n_head

# Examples
for n_embd in [384, 512, 768, 1024, 1280]:
    n_head = recommend_num_heads(n_embd)
    head_size = n_embd // n_head
    print(f"n_embd={n_embd:4d} → n_head={n_head:2d} (head_size={head_size})")

n_embd= 384 → n_head= 6 (head_size=64)
n_embd= 512 → n_head= 8 (head_size=64)
n_embd= 768 → n_head=12 (head_size=64)
n_embd=1024 → n_head=16 (head_size=64)
n_embd=1280 → n_head=20 (head_size=64)


### 4. Context Length (block_size or max_position_embeddings)

In [ ]:
import math

def estimate_memory_gb(
    n_embd: int,
    n_layer: int,
    block_size: int,
    batch_size: int = 1
) -> float:
    """
    Estimate GPU memory needed for inference.

    Rough calculation, actual may vary.
    """
    # Parameters memory (weights)
    params = 12 * n_embd**2 * n_layer
    params_memory = params * 4 / 1e9  # 4 bytes per float32

    # Activation memory (scales with batch and context)
    activations_per_token = n_layer * n_embd * 4  # Rough estimate
    activations = batch_size * block_size * activations_per_token
    activations_memory = activations * 4 / 1e9

    # Attention memory (quadratic in context!)
    attention = batch_size * n_layer * block_size**2
    attention_memory = attention * 4 / 1e9

    total = params_memory + activations_memory + attention_memory

    return total

# Compare different context lengths
n_embd, n_layer = 768, 12

print("Memory usage for 125M parameter model:\n")
for block_size in [128, 256, 512, 1024, 2048]:
    memory = estimate_memory_gb(n_embd, n_layer, block_size)
    print(f"Context {block_size:4d}: {memory:.2f} GB")

Memory usage for 125M parameter model:

Context  128: 0.36 GB
Context  256: 0.38 GB
Context  512: 0.43 GB
Context 1024: 0.54 GB
Context 2048: 0.84 GB


In [ ]:
def recommend_context_length(use_case: str) -> int:
    """Recommend context length for different use cases"""

    recommendations = {
        "classification": 128,      # Sentiment, topic classification
        "chat": 512,                # Conversational AI
        "summarization": 1024,      # Document summarization
        "qa": 512,                  # Question answering
        "code": 1024,               # Code completion
        "creative_writing": 1024,   # Story generation
        "document_analysis": 2048,  # Long documents
    }

    return recommendations.get(use_case, 512)

# Examples
for task in ["classification", "chat", "summarization", "code"]:
    length = recommend_context_length(task)
    print(f"{task:20s}: {length} tokens")

classification      : 128 tokens
chat                : 512 tokens
summarization       : 1024 tokens
code                : 1024 tokens


## Part 3: Pre-configured Architectures

Rather than choosing everything manually, start with proven configurations.

### Configuration 1: Fast Prototype (50M parameters)

In [ ]:
config_fast = {
    'vocab_size': 50257,      # GPT-2 tokenizer size
    'n_embd': 512,
    'n_layer': 8,
    'n_head': 8,
    'block_size': 256,
    'dropout': 0.1,
}

# Calculate actual parameters
def count_parameters(config):
    """Estimate total parameters from config"""
    vocab = config['vocab_size']
    embd = config['n_embd']
    layers = config['n_layer']
    block = config['block_size']

    # Embeddings
    token_embd = vocab * embd
    pos_embd = block * embd

    # Transformer layers (12 × n_embd² per layer)
    transformer = 12 * embd**2 * layers

    # Output head
    output = vocab * embd

    total = token_embd + pos_embd + transformer + output
    return total / 1e6  # Return in millions

print(f"Fast config: {count_parameters(config_fast):.1f}M parameters")

Fast config: 76.8M parameters


### Configuration 2: Balanced Medium (350M parameters)

In [ ]:
config_medium = {
    'vocab_size': 50257,
    'n_embd': 1024,
    'n_layer': 24,
    'n_head': 16,
    'block_size': 512,
    'dropout': 0.1,
}

print(f"Medium config: {count_parameters(config_medium):.1f}M parameters")

Medium config: 405.4M parameters


### Configuration 3: High Quality (1B parameters)

In [ ]:
config_large = {
    'vocab_size': 50257,
    'n_embd': 1280,
    'n_layer': 36,
    'n_head': 20,
    'block_size': 1024,
    'dropout': 0.1,
}

print(f"Large config: {count_parameters(config_large):.1f}M parameters")

Large config: 837.8M parameters


## Part 4: Building Configurable Models

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class ConfigurableGPT(nn.Module):
    """
    GPT model with flexible configuration.
    Based on Chapter 2's architecture but fully configurable.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config

        # Validate configuration
        assert config['n_embd'] % config['n_head'] == 0, \
            f"n_embd ({config['n_embd']}) must be divisible by n_head ({config['n_head']})"

        # Token + position embeddings
        self.token_embedding = nn.Embedding(config['vocab_size'], config['n_embd'])
        self.position_embedding = nn.Embedding(config['block_size'], config['n_embd'])

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config['n_layer'])
        ])

        # Output
        self.ln_f = nn.LayerNorm(config['n_embd'])
        self.lm_head = nn.Linear(config['n_embd'], config['vocab_size'], bias=False)

        # Dropout
        self.dropout = nn.Dropout(config['dropout'])

        # Initialize weights
        self.apply(self._init_weights)

        print(f"Model initialized with {self.count_parameters()/1e6:.1f}M parameters")

    def _init_weights(self, module):
        """Initialize weights (important for training stability)"""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def count_parameters(self):
        """Count total parameters"""
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None):
        """
        Forward pass.

        idx: (batch, time) tensor of token indices
        targets: (batch, time) tensor of target indices (optional)
        """
        B, T = idx.shape

        # Embeddings
        tok_emb = self.token_embedding(idx)  # (B, T, n_embd)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = self.dropout(tok_emb + pos_emb)  # (B, T, n_embd)

        # Transformer blocks
        for block in self.blocks:
            x = block(x)

        # Output
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        # Calculate loss if targets provided
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1
            )

        return logits, loss

class TransformerBlock(nn.Module):
    """Single transformer block"""

    def __init__(self, config):
        super().__init__()

        self.ln1 = nn.LayerNorm(config['n_embd'])
        self.attn = MultiHeadAttention(config)
        self.ln2 = nn.LayerNorm(config['n_embd'])
        self.mlp = FeedForward(config)

    def forward(self, x):
        # Pre-normalization (more stable than post-norm)
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention"""

    def __init__(self, config):
        super().__init__()
        assert config['n_embd'] % config['n_head'] == 0

        self.n_head = config['n_head']
        self.n_embd = config['n_embd']
        self.head_size = config['n_embd'] // config['n_head']

        # QKV projection
        self.c_attn = nn.Linear(config['n_embd'], 3 * config['n_embd'])
        # Output projection
        self.c_proj = nn.Linear(config['n_embd'], config['n_embd'])

        # Dropout
        self.attn_dropout = nn.Dropout(config['dropout'])
        self.resid_dropout = nn.Dropout(config['dropout'])

        # Causal mask
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config['block_size'], config['block_size']))
                .view(1, 1, config['block_size'], config['block_size'])
        )

    def forward(self, x):
        B, T, C = x.shape

        # Calculate Q, K, V
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # Reshape for multi-head
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)

        # Attention scores
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # Apply attention to values
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

class FeedForward(nn.Module):
    """Position-wise feed-forward network"""

    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config['n_embd'], 4 * config['n_embd']),
            nn.GELU(),  # Gaussian Error Linear Unit (better than ReLU)
            nn.Linear(4 * config['n_embd'], config['n_embd']),
            nn.Dropout(config['dropout']),
        )

    def forward(self, x):
        return self.net(x)

# Test with different configs
print("\n" + "="*60)
print("Testing different configurations:")
print("="*60)

for name, config in [('Fast', config_fast), ('Medium', config_medium), ('Large', config_large)]:
    print(f"\n{name} Config:")
    model = ConfigurableGPT(config)

    # Test forward pass
    dummy_input = torch.randint(0, config['vocab_size'], (2, 64))  # Batch=2, Seq=64
    with torch.no_grad():
        logits, _ = model(dummy_input)
    print(f"  Input shape: {dummy_input.shape}")
    print(f"  Output shape: {logits.shape}")


Testing different configurations:

Fast Config:
Model initialized with 76.8M parameters
  Input shape: torch.Size([2, 64])
  Output shape: torch.Size([2, 64, 50257])

Medium Config:
Model initialized with 405.8M parameters
  Input shape: torch.Size([2, 64])
  Output shape: torch.Size([2, 64, 50257])

Large Config:
Model initialized with 838.4M parameters
  Input shape: torch.Size([2, 64])
  Output shape: torch.Size([2, 64, 50257])


## Part 5: Decision Framework

Use this framework to choose your configuration.

### Step 1: Define Your Task

In [ ]:
class TaskProfile:
    """Profile your task requirements"""

    def __init__(
        self,
        name: str,
        complexity: str,  # 'simple', 'medium', 'complex'
        data_size_mb: int,
        sequence_length: int,  # Average input length in tokens
        quality_requirement: str,  # 'basic', 'good', 'excellent'
        speed_requirement: str,  # 'fast', 'moderate', 'slow_ok'
    ):
        self.name = name
        self.complexity = complexity
        self.data_size_mb = data_size_mb
        self.sequence_length = sequence_length
        self.quality_requirement = quality_requirement
        self.speed_requirement = speed_requirement

    def recommend_config(self):
        """Recommend configuration based on requirements"""

        # Start with base size
        if self.quality_requirement == 'basic':
            base_params = 50
        elif self.quality_requirement == 'good':
            base_params = 150
        else:  # excellent
            base_params = 350

        # Adjust for complexity
        if self.complexity == 'complex':
            base_params *= 1.5
        elif self.complexity == 'simple':
            base_params *= 0.7

        # Adjust for data availability
        if self.data_size_mb < 100:
            base_params = min(base_params, 125)

        # Choose from standard sizes
        if base_params < 75:
            return config_fast
        elif base_params < 250:
            return config_medium
        else:
            return config_large

# Example tasks
tasks = [
    TaskProfile(
        name="Email Classification",
        complexity="simple",
        data_size_mb=50,
        sequence_length=128,
        quality_requirement="good",
        speed_requirement="fast"
    ),
    TaskProfile(
        name="Medical Report Summarization",
        complexity="complex",
        data_size_mb=500,
        sequence_length=1024,
        quality_requirement="excellent",
        speed_requirement="moderate"
    ),
    TaskProfile(
        name="Code Completion",
        complexity="medium",
        data_size_mb=1000,
        sequence_length=512,
        quality_requirement="good",
        speed_requirement="fast"
    ),
]

print("\nTask-Specific Recommendations:")
print("="*60)
for task in tasks:
    config = task.recommend_config()
    params = count_parameters(config)
    print(f"\n{task.name}:")
    print(f"  Recommended: {params:.0f}M parameters")
    print(f"  n_embd={config['n_embd']}, n_layer={config['n_layer']}, "
          f"n_head={config['n_head']}, context={config['block_size']}")


Task-Specific Recommendations:

Email Classification:
  Recommended: 405M parameters
  n_embd=1024, n_layer=24, n_head=16, context=512

Medical Report Summarization:
  Recommended: 838M parameters
  n_embd=1280, n_layer=36, n_head=20, context=1024

Code Completion:
  Recommended: 405M parameters
  n_embd=1024, n_layer=24, n_head=16, context=512


### Step 2: Validate Your Choice

In [ ]:
def validate_config(config, available_gpu_gb: int, training_days_limit: int):
    """
    Check if configuration is feasible.

    Returns: (is_feasible, warnings)
    """
    warnings = []

    # Check GPU memory
    required_memory = estimate_memory_gb(
        config['n_embd'],
        config['n_layer'],
        config['block_size'],
        batch_size=8  # Typical training batch
    )

    if required_memory > available_gpu_gb:
        warnings.append(
            f"⚠️  Requires {required_memory:.1f}GB GPU memory, "
            f"you have {available_gpu_gb}GB. Reduce batch size or use gradient accumulation."
        )

    # Check training time (rough estimate)
    params_millions = count_parameters(config)
    estimated_days = params_millions / 50  # Very rough: 50M params per day on RTX 3090

    if estimated_days > training_days_limit:
        warnings.append(
            f"⚠️  Estimated training time: {estimated_days:.1f} days, "
            f"your limit: {training_days_limit} days. Consider smaller model."
        )

    # Check divisibility
    if config['n_embd'] % config['n_head'] != 0:
        warnings.append(
            f"❌ n_embd ({config['n_embd']}) must be divisible by "
            f"n_head ({config['n_head']})"
        )
        return False, warnings

    is_feasible = len([w for w in warnings if '❌' in w]) == 0
    return is_feasible, warnings

# Test validation
print("\nConfiguration Validation:")
print("="*60)

available_gpu = 12  # GB
time_limit = 3  # days

for name, config in [('Fast', config_fast), ('Medium', config_medium), ('Large', config_large)]:
    feasible, warnings = validate_config(config, available_gpu, time_limit)
    print(f"\n{name} ({count_parameters(config):.0f}M params):")
    if feasible:
        print("  ✓ Configuration is feasible")
    else:
        print("  ✗ Configuration has issues")

    for warning in warnings:
        print(f"  {warning}")


Configuration Validation:

Fast (77M params):
  ✓ Configuration is feasible

Medium (405M params):
  ✓ Configuration is feasible
  ⚠️  Estimated training time: 8.1 days, your limit: 3 days. Consider smaller model.

Large (838M params):
  ✓ Configuration is feasible
  ⚠️  Estimated training time: 16.8 days, your limit: 3 days. Consider smaller model.


## Part 6: Advanced: Custom Configurations

### Scaling Laws

There's a mathematical relationship between model size, data size, and performance:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def chinchilla_optimal_params(training_tokens_billions: float) -> float:
    """
    Chinchilla scaling law: optimal parameters for given training tokens.

    From Hoffmann et al. (2022): N_opt ≈ 20 × D
    where N = parameters, D = training tokens
    """
    return 20 * training_tokens_billions

def chinchilla_optimal_tokens(params_billions: float) -> float:
    """Optimal training tokens for given parameters"""
    return params_billions / 20

# Examples
print("\nChinchilla Optimal Scaling:")
print("="*60)

for tokens_b in [1, 5, 10, 20]:
    optimal_params = chinchilla_optimal_params(tokens_b)
    print(f"{tokens_b}B training tokens → {optimal_params:.1f}B parameters optimal")

print()

for params_m in [125, 350, 1000]:
    optimal_tokens = chinchilla_optimal_tokens(params_m / 1000)
    print(f"{params_m}M parameters → {optimal_tokens*1000:.0f}M training tokens optimal")


Chinchilla Optimal Scaling:
1B training tokens → 20.0B parameters optimal
5B training tokens → 100.0B parameters optimal
10B training tokens → 200.0B parameters optimal
20B training tokens → 400.0B parameters optimal

125M parameters → 6M training tokens optimal
350M parameters → 17M training tokens optimal
1000M parameters → 50M training tokens optimal


### Width vs Depth

In [ ]:
def compare_architectures():
    """Compare wide vs deep architectures with similar parameter counts"""

    # Target: ~350M parameters

    # Wide configuration (more n_embd, fewer layers)
    config_wide = {
        'vocab_size': 50257,
        'n_embd': 1280,
        'n_layer': 18,
        'n_head': 20,
        'block_size': 512,
        'dropout': 0.1,
    }

    # Deep configuration (smaller n_embd, more layers)
    config_deep = {
        'vocab_size': 50257,
        'n_embd': 896,
        'n_layer': 32,
        'n_head': 14,
        'block_size': 512,
        'dropout': 0.1,
    }

    print("Wide vs Deep Architectures:")
    print("="*60)
    print(f"\nWide: {count_parameters(config_wide):.0f}M params")
    print(f"  n_embd={config_wide['n_embd']}, n_layer={config_wide['n_layer']}")
    print(f"  Good for: Capturing rich representations")

    print(f"\nDeep: {count_parameters(config_deep):.0f}M params")
    print(f"  n_embd={config_deep['n_embd']}, n_layer={config_deep['n_layer']}")
    print(f"  Good for: Complex reasoning, long-range dependencies")

    print("\nGeneral guidance:")
    print("  • Wider models: Better at memorization, representations")
    print("  • Deeper models: Better at reasoning, composition")
    print("  • For most tasks: Balanced approach (medium width + depth)")

compare_architectures()

Wide vs Deep Architectures:

Wide: 483M params
  n_embd=1280, n_layer=18
  Good for: Capturing rich representations

Deep: 399M params
  n_embd=896, n_layer=32
  Good for: Complex reasoning, long-range dependencies

General guidance:
  • Wider models: Better at memorization, representations
  • Deeper models: Better at reasoning, composition
  • For most tasks: Balanced approach (medium width + depth)
